# Training Pipeline — QCNN

A **Quantum Convolutional Neural Network**, ported from
[`takh04/QCNN`](https://github.com/takh04/QCNN) (Hur, Kim & Park, *Quantum convolutional neural
network for classical data classification*, 2022). Unlike the other notebooks in this folder,
there is no PyTorch backbone to configure through `timm` — the original is a PennyLane circuit
simulated one 8-qubit statevector at a time, so the load-bearing pieces are ported as plain
`torch` tensor ops, the same departure [`vae/notebooks/quantum/qvae.ipynb`](../../../vae/notebooks/quantum/qvae.ipynb)
takes from its own Qiskit source.

**What's ported, in [`cnn/models/qcnn.py`](../../models/qcnn.py):**
- `U_SU4` (`unitary.py`) — the 15-parameter general two-qubit convolution ansatz the original's
  benchmarking run uses, built here as a dense `2^8 x 2^8` unitary matrix instead of a PennyLane
  sub-circuit.
- `Pooling_ansatz1` (`unitary.py`) — the only pooling ansatz the original ever calls
  (`CRZ; PauliX; CRX`). Pooling does **not** trace anything out — the discarded wire is simply
  never addressed again — so this port never needs a partial trace, unlike `qvae.ipynb`'s
  trash-qubit bottleneck.
- `QCNN_structure` (`QCNN_circuit.py`) — 3 conv+pool stages reducing 8 qubits to 4, to 2, to 1,
  each conv layer *reusing one shared parameter vector* across every wire pair (the circuit's
  analogue of a classical conv kernel), reading out the final surviving qubit (wire 4).

**What's replaced:** the original loops over samples one at a time and optimizes with
`NesterovMomentumOptimizer` via PennyLane's own autograd (parameter-shift under the hood). Since
every op here (unitary construction, matrix multiply, `|amplitude|^2`) is an ordinary
differentiable `torch` op, this notebook instead batches the whole circuit over the batch
dimension and trains with plain `torch.autograd` + Adam.

Data, loss shape (cross-entropy on `[P(0), P(1)]`) and the accuracy decoding rule (`P=0` if
`p[0] > p[1]` else `P=1`) match `data.py` / `Training.py` / `Benchmarking.py::accuracy_test`
exactly — see [`cnn/handlers/qcnn.py`](../../handlers/qcnn.py).

A GPU helps but is not required — the whole circuit is one `256x256` complex matrix multiply per
batch, tiny by CNN standards. `build_model` below does not require CUDA.

## Install Requirements

Only what the pipeline actually imports: `torch`/`torchvision` for the data and training loop.
No `pennylane` — the circuit is reimplemented as dense `torch` matrices, so the original's
quantum-simulation dependency is never installed.

In [ ]:
!nvidia-smi

In [ ]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

## Get the survey code

The model and handler live in this repository's `survey/src/cnn` package, so the notebook needs a
checkout of it. On Colab it clones into `/content/quantum-quantization`, or `git pull --ff-only`s
that directory if it is already there — so re-running the cell after a push picks up the new code.
Run locally, the notebook already sits inside the repo, so `find_src` climbs to `survey/src` and
git is never touched (your working tree is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `cnn` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `cnn`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / 'cnn' / 'models' / 'qcnn.py').is_file():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / 'cnn' / 'models' / 'qcnn.py').is_file(), f'qcnn.py not found under {SRC}'
print('survey src:', SRC)

## Imports

`QCNN` and the training/data helpers come from the survey's `cnn` package. `SRC` from the
previous cell goes on `sys.path`, and `os.chdir` moves into it so `./data` (the shared
[`src/data`](../../../data) folder) is where `torchvision` downloads Fashion-MNIST.

In [ ]:
import os
import sys

import torch
import torch.optim as optim
import torchvision.datasets as datasets

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m == 'cnn' or m.startswith('cnn.')]:
    del sys.modules[_m]

from cnn.models.qcnn import QCNN
from cnn.handlers.qcnn import to_amplitude_states, train_qcnn

## Configuration

`result.py`'s shipped config, kept as a plain namespace instead of `argparse`: `fashion_mnist`,
binary classes `[0, 1]` (t-shirt/top vs. trouser), the `'resize256'` amplitude embedding, `U_SU4`
convolutions, cross-entropy cost. `steps` counts optimizer **iterations**, not epochs — each step
draws one fresh minibatch sampled *with replacement*, matching `Training.py::circuit_training`.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    classes=[0, 1],                                  # binary label pair to filter Fashion-MNIST to
    lr=0.01,                                          # matches the original's Nesterov stepsize
    steps=200,                                        # optimizer iterations, not epochs
    batch_size=25,                                    # sampled with replacement each step
)

## Device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

## Dataset

Fashion-MNIST, filtered to the two classes in `args.classes` (`data.py`'s default
`dataset='fashion_mnist'`, `classes=[0, 1]`). Each 28x28 image is bilinear-resized to a
256 = 2^8-length vector and L2-normalized (`to_amplitude_states`, the `'resize256'` mode feeding
`AmplitudeEmbedding` in the original) so it is a valid 8-qubit statevector.

In [ ]:
train_raw = datasets.FashionMNIST(root='./data', train=True, download=True)
test_raw = datasets.FashionMNIST(root='./data', train=False, download=True)

train_mask = torch.isin(train_raw.targets, torch.tensor(args.classes))
test_mask = torch.isin(test_raw.targets, torch.tensor(args.classes))

label_map = {c: i for i, c in enumerate(args.classes)}   # remap to {0, 1}
train_x = to_amplitude_states(train_raw.data[train_mask])
train_y = torch.tensor([label_map[int(t)] for t in train_raw.targets[train_mask]])
test_x = to_amplitude_states(test_raw.data[test_mask])
test_y = torch.tensor([label_map[int(t)] for t in test_raw.targets[test_mask]])

print('train:', train_x.shape, train_y.shape)
print('test: ', test_x.shape, test_y.shape)

## Model

`QCNN()` builds the fixed 8-qubit conv+pool topology described above, with `float64` parameters
(unitary/amplitude math is numerically sensitive, matching `qvae.ipynb`'s own choice of dtype).
`forward` returns `[P(0), P(1)]` marginal probabilities of the final surviving qubit (wire 4),
matching `qml.probs(wires=4)` in the original's cross-entropy mode.

In [ ]:
net = QCNN().to(device)
print(net)
print('trainable parameters:', sum(p.numel() for p in net.parameters()))

## Optimizer

Plain Adam in place of the original's `NesterovMomentumOptimizer` — the circuit is fully
differentiable end to end (dense unitary matmuls + `|amplitude|^2`), so ordinary backprop replaces
PennyLane's parameter-shift gradients without changing what's being optimized.

In [ ]:
optimizer = optim.Adam(net.parameters(), lr=args.lr)

## Train

`train_qcnn` (from the handler) runs `args.steps` minibatch iterations — each step samples
`args.batch_size` training images with replacement, exactly `Training.py::circuit_training` — and
reports test accuracy once training finishes.

In [ ]:
test_acc = train_qcnn(net, optimizer, train_x, train_y, test_x, test_y,
                      steps=args.steps, batch_size=args.batch_size, device=device)

## Evaluate

`predict` decodes each sample's `[P(0), P(1)]` the same way `Benchmarking.py::accuracy_test`
does — class `0` if `P(0) > P(1)` else `1` — so per-sample predictions on the test set can be
inspected directly.

In [ ]:
from cnn.handlers.qcnn import predict

net.eval()
with torch.no_grad():
    test_probs = net(test_x.to(device))
    preds = predict(test_probs).cpu()

print('predictions:', preds[:20].tolist())
print('labels:     ', test_y[:20].tolist())
print(f'test accuracy: {(preds == test_y).float().mean().item():.4f}')